# Model Training

This notebook trains the coding agent model on the prepared dataset.

**Training Process:**
1. Load preprocessed data
2. Create data loaders
3. Initialize model and optimizer
4. Training loop with validation
5. Save best model

In [1]:
#!pip install torch

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    T5ForConditionalGeneration,
    RobertaTokenizer,
    #AdamW,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
import pandas as pd
import json
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from datetime import datetime

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set paths
PROCESSED_DATA_DIR = Path('../data/processed')
MODEL_DIR = Path('../models').resolve()
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

Using device: cpu


## 1. Load Configuration and Data

In [3]:
# Load training configuration
with open(MODEL_DIR / 'training_config.json', 'r') as f:
    config = json.load(f)

print("Training Configuration:")
for key, value in config.items():
    print(f"  {key}: {value}")

# Load datasets
train_data = pd.read_json(PROCESSED_DATA_DIR / 'train.json')
val_data = pd.read_json(PROCESSED_DATA_DIR / 'validation.json')

print(f"\nTrain samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")

Training Configuration:
  model_name: Salesforce/codet5-base
  max_input_length: 512
  max_output_length: 512
  num_epochs: 10
  batch_size: 4
  learning_rate: 5e-05
  weight_decay: 0.01
  warmup_steps: 500
  num_beams: 4
  temperature: 0.7
  top_p: 0.95
  gradient_accumulation_steps: 4
  max_grad_norm: 1.0
  eval_steps: 100
  save_steps: 500
  logging_steps: 50
  early_stopping_patience: 3
  early_stopping_threshold: 0.001
  use_lora: True
  lora_r: 16
  lora_alpha: 32
  lora_dropout: 0.05
  lora_target_modules: ['q', 'v']
  fp16: True
  bf16: False

Train samples: 700
Validation samples: 300


## 2. Create Custom Dataset Class

In [4]:
class CodeDataset(Dataset):
    def __init__(self, data, tokenizer, max_input_length, max_output_length):
        self.data = data
        self.tokenizer = tokenizer
        self.max_input_length = max_input_length
        self.max_output_length = max_output_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        
        # Tokenize input
        input_encoding = self.tokenizer(
            row['input'],
            max_length=self.max_input_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        # Tokenize output
        output_encoding = self.tokenizer(
            row['output'],
            max_length=self.max_output_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        # Prepare labels (replace padding token id with -100 for loss calculation)
        labels = output_encoding['input_ids'].clone()
        labels[labels == self.tokenizer.pad_token_id] = -100
        
        return {
            'input_ids': input_encoding['input_ids'].squeeze(),
            'attention_mask': input_encoding['attention_mask'].squeeze(),
            'labels': labels.squeeze()
        }

print("Dataset class defined")

Dataset class defined


## 3. Initialize Model and Tokenizer

In [5]:
# Load tokenizer and model
model_name = config['model_name']
tokenizer = RobertaTokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)
model = model.to(device)

print(f"Model loaded: {model_name}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Model loaded: Salesforce/codet5-base
Parameters: 222,882,048
Trainable parameters: 222,882,048


## 4. Create Data Loaders

In [6]:
# Create datasets
train_dataset = CodeDataset(
    train_data,
    tokenizer,
    config['max_input_length'],
    config['max_output_length']
)

val_dataset = CodeDataset(
    val_data,
    tokenizer,
    config['max_input_length'],
    config['max_output_length']
)

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=config['batch_size'],
    shuffle=True,
    num_workers=0  # Set to 0 for Windows compatibility
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config['batch_size'],
    shuffle=False,
    num_workers=0
)

print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

# Test data loader
sample_batch = next(iter(train_loader))
print(f"\nSample batch shapes:")
print(f"  Input IDs: {sample_batch['input_ids'].shape}")
print(f"  Attention mask: {sample_batch['attention_mask'].shape}")
print(f"  Labels: {sample_batch['labels'].shape}")

Train batches: 175
Validation batches: 75

Sample batch shapes:
  Input IDs: torch.Size([4, 512])
  Attention mask: torch.Size([4, 512])
  Labels: torch.Size([4, 512])


## 5. Setup Optimizer and Scheduler

In [7]:
# Calculate total training steps
num_epochs = config['num_epochs']
total_steps = len(train_loader) * num_epochs

# Initialize optimizer
optimizer = AdamW(
    model.parameters(),
    lr=config['learning_rate'],
    weight_decay=config['weight_decay']
)

# Initialize learning rate scheduler
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=config['warmup_steps'],
    num_training_steps=total_steps
)

print(f"Optimizer: AdamW")
print(f"Learning rate: {config['learning_rate']}")
print(f"Total training steps: {total_steps}")
print(f"Warmup steps: {config['warmup_steps']}")

Optimizer: AdamW
Learning rate: 5e-05
Total training steps: 1750
Warmup steps: 500


## 6. Training Functions

In [8]:
def train_epoch(model, train_loader, optimizer, scheduler, device, grad_accum_steps):
    """Train for one epoch"""
    model.train()
    total_loss = 0
    progress_bar = tqdm(train_loader, desc="Training")
    
    optimizer.zero_grad()
    
    for step, batch in enumerate(progress_bar):
        # Move batch to device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        # Forward pass
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        
        loss = outputs.loss / grad_accum_steps
        total_loss += loss.item() * grad_accum_steps
        
        # Backward pass
        loss.backward()
        
        # Update weights every grad_accum_steps
        if (step + 1) % grad_accum_steps == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), config['max_grad_norm'])
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
        
        # Update progress bar
        progress_bar.set_postfix({'loss': loss.item() * grad_accum_steps})
    
    return total_loss / len(train_loader)

def evaluate(model, val_loader, device):
    """Evaluate on validation set"""
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        progress_bar = tqdm(val_loader, desc="Evaluating")
        for batch in progress_bar:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            
            total_loss += outputs.loss.item()
            progress_bar.set_postfix({'loss': outputs.loss.item()})
    
    return total_loss / len(val_loader)

print("Training functions defined")

Training functions defined


## 7. Training Loop

In [ ]:
# Training history
history = {
    'train_loss': [],
    'val_loss': [],
    'learning_rate': []
}

best_val_loss = float('inf')
patience_counter = 0

print("Starting training...\n")
start_time = datetime.now()

for epoch in range(num_epochs):
    print(f"\n{'='*60}")
    print(f"Epoch {epoch + 1}/{num_epochs}")
    print(f"{'='*60}")
    
    # Train
    train_loss = train_epoch(
        model,
        train_loader,
        optimizer,
        scheduler,
        device,
        config['gradient_accumulation_steps']
    )
    
    # Evaluate
    val_loss = evaluate(model, val_loader, device)
    
    # Get current learning rate
    current_lr = optimizer.param_groups[0]['lr']
    
    # Save history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['learning_rate'].append(current_lr)
    
    # Print epoch summary
    print(f"\nEpoch {epoch + 1} Summary:")
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Val Loss: {val_loss:.4f}")
    print(f"  Learning Rate: {current_lr:.2e}")
    
    # Save best model
    if val_loss < best_val_loss:
        improvement = best_val_loss - val_loss
        print(f"  ✓ Validation loss improved by {improvement:.4f}")
        best_val_loss = val_loss
        patience_counter = 0
        
        # Save model
        model_save_path = MODEL_DIR / 'best_model'
        model.save_pretrained(model_save_path)
        tokenizer.save_pretrained(model_save_path)
        print(f"  ✓ Model saved to {model_save_path}")
    else:
        patience_counter += 1
        print(f"  ✗ No improvement (patience: {patience_counter}/{config['early_stopping_patience']})")
    
    # Early stopping
    if patience_counter >= config['early_stopping_patience']:
        print(f"\nEarly stopping triggered after {epoch + 1} epochs")
        break

end_time = datetime.now()
training_duration = end_time - start_time

print(f"\n{'='*60}")
print("TRAINING COMPLETE!")
print(f"{'='*60}")
print(f"Total training time: {training_duration}")
print(f"Best validation loss: {best_val_loss:.4f}")

Starting training...


Epoch 1/10


Training:   0%|          | 0/175 [00:00<?, ?it/s]

## 8. Save Training History

In [ ]:
# Save training history
history_path = MODEL_DIR / 'training_history.json'
with open(history_path, 'w') as f:
    json.dump(history, f, indent=2)

print(f"Training history saved to: {history_path}")

## 9. Visualize Training Progress

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss curves
epochs_range = range(1, len(history['train_loss']) + 1)
axes[0].plot(epochs_range, history['train_loss'], 'b-', label='Train Loss', linewidth=2)
axes[0].plot(epochs_range, history['val_loss'], 'r-', label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Learning rate schedule
axes[1].plot(epochs_range, history['learning_rate'], 'g-', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Learning Rate', fontsize=12)
axes[1].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].ticklabel_format(style='scientific', axis='y', scilimits=(0,0))

plt.tight_layout()
plt.savefig(MODEL_DIR / 'training_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Training curves saved to: {MODEL_DIR / 'training_curves.png'}")

## 10. Test Trained Model

In [ ]:
# Load best model
best_model_path = (MODEL_DIR / 'best_model').resolve()
# Check if model exists first to avoid HFValidationError on Windows
if not best_model_path.exists():
    raise FileNotFoundError(f"Model not found at {best_model_path}. Please train the model first.")
trained_model = T5ForConditionalGeneration.from_pretrained(best_model_path.as_posix())
trained_model = trained_model.to(device)
trained_model.eval()

print("Best model loaded for testing\n")

# Test with sample prompts
test_prompts = [
    "Language: csharp\nFramework: dotnet8\nTask: Create a simple API controller with GET endpoint",
    "Language: typescript\nFramework: angular\nTask: Create an Angular service with HTTP client",
    "Language: sql\nFramework: mssql\nTask: Create a table with primary key and foreign key"
]

for i, prompt in enumerate(test_prompts, 1):
    print(f"{'='*60}")
    print(f"Test {i}")
    print(f"{'='*60}")
    print(f"Prompt:\n{prompt}\n")
    
    # Tokenize
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=config['max_input_length'],
        truncation=True
    ).to(device)
    
    # Generate
    with torch.no_grad():
        outputs = trained_model.generate(
            inputs['input_ids'],
            max_length=config['max_output_length'],
            num_beams=config['num_beams'],
            temperature=config['temperature'],
            top_p=config['top_p'],
            early_stopping=True
        )
    
    # Decode
    generated_code = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"Generated Code:\n{generated_code}\n")

print("\n" + "="*60)
print("TRAINING COMPLETE!")
print("="*60)
print(f"\nBest model saved at: {best_model_path}")
print(f"Next step: Run notebook 05_evaluation.ipynb to evaluate the model")